# Accessing Data

In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd aus_substation_electricity/

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

## Holiday Function

In [ ]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

In [ ]:
# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

# Spaghetti Plot
- similar to confidence interval plot of anomaly
- creating an anomaly plot of the 30 +/- days either side of public holiday
- the prominate line will be the public holiday 24 hours, and the greyed lines are the days either side

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def compare_holiday_only_window_spaghetti(
    demand,
    info,
    station,
    years,
    holiday_func,
    holiday_name,
):
    """
    Spaghetti plot version:
    - Grey lines = each individual day in the ±30‑day window (all years)
    - Red line   = mean 24‑hour anomaly profile for the holiday
    """

    # --- Validate input ---
    if not (isinstance(years, tuple) and len(years) == 2):
        raise ValueError("`years` must be a tuple like (2006, 2007).")

    start_year, end_year = years
    year_list = list(range(start_year, end_year + 1))

    # --- Prepare hourly demand data ---
    demand.index = pd.to_datetime(demand.index)
    hourly = demand[[station]].resample("h").mean()

    # --- Collect ±30‑day windows around the holiday for each year ---
    windows = []
    for yr in year_list:
        ref_date = holiday_func(yr)
        start = ref_date - pd.Timedelta(days=30)
        end   = ref_date + pd.Timedelta(days=30)
        windows.append(hourly.loc[start:end].copy())

    combined = pd.concat(windows)

    # --- Compute baseline stratified by hour ---
    combined["hour"] = combined.index.hour
    baseline_by_hour = combined.groupby("hour")[station].mean()

    # --- Compute anomalies ---
    anomalies = combined.copy()
    anomalies["anomaly"] = anomalies[station] - anomalies["hour"].map(baseline_by_hour)

    # --- Build daily anomaly curves for ALL days ---
    daily_curves = []
    for date, group in anomalies.groupby(anomalies.index.date):
        # Extract 24-hour curve
        day = pd.date_range(pd.Timestamp(date), periods=24, freq="h")
        curve = anomalies["anomaly"].reindex(day)
        curve = curve.interpolate(limit_direction="both")
        curve.index = range(24)
        daily_curves.append(curve.rename(pd.Timestamp(date)))

    daily_matrix = pd.concat(daily_curves, axis=1)

    # --- Extract holiday-only curves ---
    holiday_hours = []
    for yr in year_list:
        ref_date = holiday_func(yr)
        hours = pd.date_range(ref_date, ref_date + pd.Timedelta(hours=23), freq="h")
        curve = anomalies["anomaly"].reindex(hours)
        curve = curve.interpolate(limit_direction="both")
        curve.index = range(24)
        holiday_hours.append(curve.rename(yr))

    holiday_matrix = pd.concat(holiday_hours, axis=1)
    holiday_profile = holiday_matrix.mean(axis=1)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(12, 5))

    # Grey lines = each day in ±30-day window
    for col in daily_matrix.columns:
        ax.plot(
            daily_matrix.index,
            daily_matrix[col],
            color="gray",
            alpha=0.25,
            linewidth=0.8
        )

    # Red line = mean holiday profile
    ax.plot(
        holiday_profile.index,
        holiday_profile.values,
        color="red",
        linewidth=2.5,
        marker="o",
        label=f"{holiday_name} Mean Profile"
    )

    # Formatting
    ax.axhline(0, color="black", linewidth=1)
    ax.grid(axis='y', linestyle='-', linewidth=0.5, color='gray', alpha=0.3)
    ax.set_xticks(np.arange(0, 24))
    ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45)

    full_name = info.loc[station, "Name"]
    ax.set_title(
        f"{full_name} Demand Anomaly: {holiday_name} ({start_year}–{end_year}, ±30‑Day Window)",
        fontsize=14
    )
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Electricity Demand Anomaly")
    ax.legend()
    fig.tight_layout()

    plt.show(fig)

In [ ]:
#defining yearly intervals I want to generate plots for
year_pairs = [
    (2004, 2005),
    (2005, 2006),
    (2016, 2017),
    (2017, 2018),
]

In [ ]:
compare_holiday_only_window_spaghetti(
    demand=demand,
    info=info,
    station="BLAKE",
    years=(2006,2007),
    holiday_func=lambda y: pd.Timestamp(y, 12, 25),
    holiday_name="Christmas Day",
)

In [ ]:
import os

SAVE_DIR = "/home/565/pv3484/aus_substation_electricity/figures/anomaly_demand/confidence_interval"
os.makedirs(SAVE_DIR, exist_ok=True)

# Your chosen 2-year windows
year_pairs = [
    (2004, 2005),
    (2005, 2006),
    (2016, 2017),
    (2017, 2018),
]

# First 3 substations
stations = info.index[:3]

for holiday_name, holiday_func in HOLIDAYS_VIC.items():
    print(f"Processing holiday: {holiday_name}")

    for station in stations:
        print(f"  Station: {station}")

        for (y1, y2) in year_pairs:
            print(f"    Years: {y1}-{y2}")

            fig = compare_holiday_only_window_spaghetti(
                demand=demand,
                info=info,
                station=station,
                years=(y1, y2),
                holiday_func=holiday_func,
                holiday_name=holiday_name
            )

            filename = f"{station}_{holiday_name.replace(' ', '_')}_{y1}_{y2}.png"
            filepath = os.path.join(SAVE_DIR, filename)

            fig.savefig(filepath, dpi=200)